# 🎣 Phishing Website Detection — EDA & Model Notebook

**Author:** Your Name  
**Goal:** Classify URLs as *Phishing* or *Legitimate* using structured URL features.

---
### Sections
1. Setup & Imports
2. Load & Inspect Data
3. Exploratory Data Analysis (EDA)
4. Feature Engineering
5. Train / Test Split & Scaling
6. Model Training (Random Forest)
7. Evaluation
8. Feature Importance
9. Single Prediction Demo

In [ ]:
# ── 1. Setup & Imports ──────────────────────────────────────────────────────
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, accuracy_score, f1_score
)
import joblib

sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline
print('✅ All imports successful')

In [ ]:
# ── 2. Load & Inspect Data ──────────────────────────────────────────────────
df = pd.read_csv('../data/phishing_dataset.csv')
print(f'Shape: {df.shape}')
df.head(10)

In [ ]:
# Basic statistics
df.describe().T.style.background_gradient(cmap='Blues')

In [ ]:
# Missing values & dtypes
print('Missing values per column:')
print(df.isnull().sum())
print('\nData types:')
print(df.dtypes)

In [ ]:
# ── 3. EDA ──────────────────────────────────────────────────────────────────

# 3a. Class distribution
fig, ax = plt.subplots(figsize=(5, 4))
counts = df['label'].value_counts()
ax.bar(['Legitimate', 'Phishing'], counts.values,
       color=['#2ecc71', '#e74c3c'], width=0.5, edgecolor='white')
for i, v in enumerate(counts.values):
    ax.text(i, v + 10, str(v), ha='center', fontweight='bold')
ax.set_title('Class Distribution', fontsize=14, fontweight='bold')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
# 3b. Box plots — key features by class
key_feats = ['url_length', 'num_dots', 'entropy', 'domain_age_days', 'subdomain_count']
fig, axes = plt.subplots(1, len(key_feats), figsize=(18, 5))
for ax, feat in zip(axes, key_feats):
    df.boxplot(column=feat, by='label', ax=ax,
               boxprops=dict(color='#2980b9'),
               medianprops=dict(color='#e74c3c', linewidth=2))
    ax.set_title(feat, fontsize=11)
    ax.set_xlabel('')
    ax.set_xticklabels(['Legit', 'Phishing'])
plt.suptitle('Feature Distributions by Class', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# 3c. Correlation heatmap
fig, ax = plt.subplots(figsize=(12, 9))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, linewidths=0.5, ax=ax)
ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── 4. Feature Engineering ───────────────────────────────────────────────────
df['dots_per_length'] = df['num_dots'] / (df['url_length'] + 1)
df['digit_ratio']     = df['num_digits'] / (df['url_length'] + 1)
df['suspicion_score'] = (
    df['has_ip'].astype(int)
    + df['tld_suspicious'].astype(int)
    + (1 - df['has_https'].astype(int))
    + (df['num_at'] > 0).astype(int)
    + (df['num_hyphens'] > 3).astype(int)
)
print('✅ New features added: dots_per_length, digit_ratio, suspicion_score')
df[['dots_per_length', 'digit_ratio', 'suspicion_score', 'label']].head()

In [ ]:
# ── 5. Split & Scale ─────────────────────────────────────────────────────────
FEATURES = [
    'url_length', 'num_dots', 'num_hyphens', 'num_at', 'num_question',
    'num_ampersand', 'num_digits', 'has_https', 'has_ip', 'subdomain_count',
    'path_length', 'entropy', 'tld_suspicious', 'domain_age_days',
    'dots_per_length', 'digit_ratio', 'suspicion_score'
]
TARGET = 'label'

X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Train: {X_train_sc.shape}  Test: {X_test_sc.shape}')

In [ ]:
# ── 6. Model Training ────────────────────────────────────────────────────────
rf = RandomForestClassifier(
    n_estimators=200, max_depth=15, min_samples_split=5,
    class_weight='balanced', random_state=42, n_jobs=-1
)

# 5-Fold Cross Validation
cv_scores = cross_val_score(rf, X_train_sc, y_train, cv=5, scoring='f1')
print(f'CV F1 (5-fold): {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

rf.fit(X_train_sc, y_train)
print('✅ Model trained')

In [ ]:
# ── 7. Evaluation ────────────────────────────────────────────────────────────
y_pred  = rf.predict(X_test_sc)
y_proba = rf.predict_proba(X_test_sc)[:, 1]

print(f'Accuracy : {accuracy_score(y_test, y_pred):.4f}')
print(f'F1 Score : {f1_score(y_test, y_pred):.4f}')
print(f'ROC-AUC  : {roc_auc_score(y_test, y_proba):.4f}')
print('\n', classification_report(y_test, y_pred, target_names=['Legitimate','Phishing']))

In [ ]:
# Confusion Matrix
fig, ax = plt.subplots(figsize=(6, 5))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Legitimate','Phishing'],
            yticklabels=['Legitimate','Phishing'], ax=ax)
ax.set_title('Confusion Matrix', fontsize=14, fontweight='bold')
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
plt.tight_layout(); plt.show()

In [ ]:
# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_proba)
auc = roc_auc_score(y_test, y_proba)
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(fpr, tpr, lw=2, color='#2980b9', label=f'AUC = {auc:.4f}')
ax.plot([0,1],[0,1],'k--', lw=1)
ax.fill_between(fpr, tpr, alpha=0.08, color='#2980b9')
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve', fontsize=14, fontweight='bold')
ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
# ── 8. Feature Importance ─────────────────────────────────────────────────────
importances = rf.feature_importances_
indices = np.argsort(importances)[::-1][:15]
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh([FEATURES[i] for i in reversed(indices)],
        importances[indices[::-1]], color='#2980b9', edgecolor='white')
ax.set_title('Top-15 Feature Importances', fontsize=14, fontweight='bold')
ax.set_xlabel('Importance Score')
plt.tight_layout(); plt.show()

In [ ]:
# ── 9. Single Prediction Demo ─────────────────────────────────────────────────
# Simulated phishing URL features
phishing_sample = {
    'url_length': 135, 'num_dots': 7, 'num_hyphens': 5, 'num_at': 2,
    'num_question': 3, 'num_ampersand': 4, 'num_digits': 10, 'has_https': 0,
    'has_ip': 1, 'subdomain_count': 4, 'path_length': 80, 'entropy': 4.8,
    'tld_suspicious': 1, 'domain_age_days': 7,
    'dots_per_length': 7/136, 'digit_ratio': 10/136, 'suspicion_score': 5
}

vector = np.array([[phishing_sample[f] for f in FEATURES]])
vector_sc = scaler.transform(vector)
prob = rf.predict_proba(vector_sc)[0][1]
pred = int(prob >= 0.5)

print('── Single URL Prediction ──')
print(f'Phishing probability : {prob:.2%}')
print(f'Result               : {"⚠️  PHISHING" if pred else "✅ LEGITIMATE"}')

In [ ]:
# Save artifacts from notebook
os.makedirs('../models', exist_ok=True)
joblib.dump(rf,     '../models/phishing_model.pkl')
joblib.dump(scaler, '../models/scaler.pkl')
print('✅ Model and scaler saved to models/')